# Reverse-CALM Prototype Test

This notebook implements the exact testing steps from Section 24 and 39 of the handoff document.
It tests a single example to verify that the FP32 fix prevents NaNs during the forward/backward/optimizer step.

In [ ]:
import sys
import torch
import json
from transformers import AutoTokenizer

# Ensure Kaggle environment can find our repo
REPO_PATH = "/kaggle/working/reverse-calm"
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

from model.calm import CALM, CALMConfig

In [ ]:
# 1. Fresh Model Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

config = CALMConfig(
    anchor_model="nischay185/konkani-qwen2-1.5b",
    aug_model="deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
    num_connections=4,
    num_heads=4,
)

# 2. Recreate model explicitly to ensure updated layers.py is used
model = CALM(config)

# 3. Memory Strategy (DeepSeek CPU, Qwen GPU, Bridge GPU FP32)
model.aug_model = model.aug_model.to("cpu")
model.anchor_model = model.anchor_model.to(device)
model.cross_attention_hooks = model.cross_attention_hooks.to(device)
model.cross_attention_hooks.float() # Keep bridge FP32

# 4. Freeze DeepSeek and Qwen
for param in model.aug_model.parameters():
    param.requires_grad = False
for param in model.anchor_model.parameters():
    param.requires_grad = False
for param in model.cross_attention_hooks.parameters():
    param.requires_grad = True

model.aug_model.eval()
model.anchor_model.eval()
model.train() # Set bridge to train

print("Model loaded and frozen appropriately.")

In [ ]:
# 5. Setup tokenizers and load one example
anchor_tokenizer = AutoTokenizer.from_pretrained(config.anchor_model)
aug_tokenizer = AutoTokenizer.from_pretrained(config.aug_model)

# We will use a dummy example mirroring the dataset since we might not have the dataset.jsonl mounted locally right now
example = {
  "question_english": "A farmer has 10 mangoes and sells 4. How many remain?",
  "reasoning": "The farmer starts with 10 mangoes and sells 4. Therefore, 10 - 4 = 6 mangoes remain.",
  "answer_konkani": "6 आंबे उरतात."
}

reasoning_text = (
    f"Question: {example['question_english']}\n\n"
    f"Reasoning: {example['reasoning']}"
)
target_text = example["answer_konkani"]

aug_inputs = aug_tokenizer(
    reasoning_text,
    return_tensors="pt",
    truncation=True,
    max_length=512,
)

target_inputs = anchor_tokenizer(
    target_text,
    return_tensors="pt",
    truncation=True,
    max_length=128,
)

# Move inputs to correct devices
aug_inputs = {k: v.to("cpu") for k, v in aug_inputs.items()}
target_inputs = {k: v.to(device) for k, v in target_inputs.items()}

# Teacher forcing
qwen_input_ids = target_inputs["input_ids"][:, :-1]
qwen_attention_mask = target_inputs["attention_mask"][:, :-1]
qwen_labels = target_inputs["input_ids"][:, 1:].clone()


In [ ]:
# Setup Optimizer
optimizer = torch.optim.AdamW(
    model.cross_attention_hooks.parameters(),
    lr=1e-4,
)
optimizer.zero_grad()

# 6. First Forward
print("Running first forward pass...")
outputs = model(
    input_ids=qwen_input_ids,
    attention_mask=qwen_attention_mask,
    aug_input_ids=aug_inputs["input_ids"],
    aug_attention_mask=aug_inputs["attention_mask"],
    labels=qwen_labels,
    use_cache=False,
)

loss = outputs.loss
print(f"Loss 1: {loss.item()}")

# 7. Verify finite loss
assert torch.isfinite(loss).all(), f"Non-finite loss: {loss.item()}"
print("[✓] Forward loss is finite")


In [ ]:
# 8. Backward
print("Running backward pass...")
loss.backward()

# 9. Verify finite gradients
for name, param in model.named_parameters():
    if param.requires_grad:
        assert param.grad is not None, f"No gradient: {name}"
        assert torch.isfinite(param.grad).all(), f"Non-finite gradient: {name}"
print("[✓] All gradients are finite")

# 10. Optimizer Step
print("Running optimizer step...")
optimizer.step()

# 11. Verify finite parameters
for name, param in model.named_parameters():
    if param.requires_grad:
        assert torch.isfinite(param).all(), f"Non-finite parameter after update: {name}"
print("[✓] All bridge parameters are finite after update")


In [ ]:
# 12. Second Forward
optimizer.zero_grad()
print("Running second forward pass...")
outputs2 = model(
    input_ids=qwen_input_ids,
    attention_mask=qwen_attention_mask,
    aug_input_ids=aug_inputs["input_ids"],
    aug_attention_mask=aug_inputs["attention_mask"],
    labels=qwen_labels,
    use_cache=False,
)

loss2 = outputs2.loss
print(f"Loss 2: {loss2.item()}")

# 13. Verify finite loss again
assert torch.isfinite(loss2).all(), f"Non-finite loss on second pass: {loss2.item()}"
print("[✓] Second forward loss is finite")
print("\nSUCCESS: The numerical stability test passed!")
